# Microproyecto 2: Identificación de Relaciones Semánticas con PLN y ML

---
## Enunciado

---
### A. Objetivo
Desarrollar una solución basada en técnicas de **procesamiento de lenguaje natural (PLN)** y **machine learning (ML)** que facilite la interpretación y análisis de información textual para la identificación de relaciones semánticas con los **Objetivos de Desarrollo Sostenible (ODS)**.

### B. Conjunto de Datos
El conjunto de datos forma parte del proyecto [OSDG Community Dataset (OSDG-CD)](https://osdg.ai/news/New-release-of-OSDG-Community-dataset) en su versión 2023, que contiene un total de **40.067 textos**, de los cuales **3.000** provienen de fuentes relacionadas con las Naciones Unidas. También contiene documentos públicos, resúmenes de artículos y reportes.

La plataforma reúne investigadores, expertos en la materia y defensores de los ODS de todo el mundo para crear una fuente amplia y precisa de información textual sobre los ODS. Los voluntarios de la comunidad utilizan la plataforma para participar en ejercicios de etiquetado en los que validan la relevancia de cada texto para los ODS basándose en sus conocimientos previos.

> **Traducción y Aumentación de Datos:**
> - Los textos utilizados en este proyecto han sido traducidos al español mediante herramientas como [DeepL](https://www.deepl.com/es/translator).
> - Se realizó aumentación de textos a través de la API de [ChatGPT / OpenAI](https://chat.openai.com/g/g-I1XNbsyDK-api-docs).

### C. Actividades a Realizar

1. **Preparación de los textos:**
   - Utilizar el esquema de bolsa de palabras (**BoW**) con pesado **TF-IDF**.
   - Construir un **pipeline** que integre todas las transformaciones y preprocesamientos que se consideren adecuados.

2. **Modelado de Tópicos (LSA):**
   - A partir de la matriz TF-IDF construida, aplicar el algoritmo SVD truncado (`TruncatedSVD` de scikit-learn) para obtener un modelo de tópicos mediante **Análisis Semántico Latente (LSA)**.
   - Explorar un número reducido de componentes (por ejemplo, entre 10 y 20).
   - Para al menos **5 componentes**, identificar y mostrar las palabras con mayor peso a modo de *tópicos*.
   - Interpretar cualitativamente si estos tópicos guardan relación con algunos de los 17 ODS trabajados en el proyecto.

3. **Desarrollo del Modelo de Clasificación:**
   - Construir un modelo de clasificación que permita relacionar un texto con su respectivo ODS.
   - Para manejar la complejidad del espacio de entrada, se puede reutilizar la descomposición SVD (LSA) o aplicar otra técnica de reducción de dimensionalidad pertinente.

4. **Evaluación del Modelo:**
   - Evaluar el modelo con un conjunto de prueba (textos no utilizados durante la etapa de entrenamiento/aprendizaje).

### D. Consideraciones
El algoritmo de clasificación a utilizar, así como la técnica de reducción de la dimensionalidad, queda a consideración de cada grupo; sin embargo, **es fundamental justificar la elección de cada técnica**.

### E. Entregable
- **Archivos:** Notebook en formatos `.ipynb` y `.html` con el método desarrollado.
- **Documentación:** El notebook debe estar completamente documentado con las justificaciones de las decisiones tomadas en cada paso.
- **Ejecución visible:** Deben ser visibles las salidas y ejecuciones de cada celda.
- **Evidencia práctica:** Para evidenciar el desempeño del método construido, el notebook debe mostrar las clasificaciones para al menos **4 textos del conjunto de test**.
- **Plazo:** Entrega al final de la **Semana 7** en el espacio correspondiente.

### F. Criterios de Evaluación

| Actividad | Porcentaje |
| :--- | :---: |
| **Preparación de los datos**, incluyendo la reducción de la dimensionalidad y justificación de decisiones tomadas. | **30%** |
| **Construcción del pipeline** de preparación de datos. | **15%** |
| **Construcción del modelo de clasificación** con el algoritmo seleccionado, búsqueda de hiperparámetros y validación con medidas de evaluación adecuadas (justificando algoritmo, métricas y reducción de dimensionalidad). | **30%** |
| **Evidencia del desempeño** del modelo mostrando clasificaciones sobre un conjunto de textos no utilizados durante el aprendizaje. | **10%** |
| **Construcción del modelo LSA** sobre la matriz TF-IDF e interpretación cualitativa de al menos 5 tópicos frente a los ODS. | **15%** |
| **Total** | **100%** |

### G. Bonificación Adicional (Opcional — +15 Puntos)
Con el propósito de fortalecer las competencias en despliegue y aplicación práctica de modelos de ML, se otorgará una bonificación adicional de **15 puntos** a los grupos que implementen el modelo en una aplicación interactiva utilizando **Streamlit**.

#### Requisitos de la Aplicación:
- Permitir al usuario ingresar un texto libre.
- Procesar el texto utilizando el mismo pipeline construido en el proyecto.
- Generar como salida la predicción del Objetivo de Desarrollo Sostenible (ODS) correspondiente.
- Ser totalmente funcional y ejecutarse correctamente.

> *Nota:* La bonificación es voluntaria y no reemplaza los criterios de la rúbrica; premia el paso del entorno experimental al despliegue práctico.

---
## Solución

---

### Importe Librerias

In [1]:
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, SnowballStemmer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC
from stop_words import get_stop_words

import joblib
import pandas as pd
import re as regexExpression
import string
import warnings

warnings.filterwarnings('ignore')

### Carga de datos

In [2]:
raw_data = pd.read_excel('data/Datos_textosODS.xlsx')
raw_data.sample(5)

,textos,ODS
3287,"Por ello, los países de la región deben redobl...",1
7112,Siguiendo las disposiciones de la Convención d...,5
7260,"En segundo lugar, la Institución de Seguridad ...",3
4806,La ley anterior tenía una serie de deficiencia...,3
4594,La inmensa mayoría (98%) de las personas desnu...,2


Validamos si existen datos duplicados o nulos:

In [3]:
print(f'Datos duplicados:\n{raw_data.duplicated().sum()}')
print(f'Datos nulos:\n{raw_data.isna().sum()}')

Datos duplicados:
0
Datos nulos:
textos    0
ODS       0
dtype: int64


No hay neceesidad de eliminar datos nulos o duplicados.

### Funcion para preparar textos

In [4]:
stop_words_es = set(stopwords.words('spanish'))

def prepare_text(text):
    tokenizer = RegexpTokenizer(r'\w+')
    stemmer = SnowballStemmer(language='spanish')
    tokens = tokenizer.tokenize(text.lower())
    tokens = [word for word in tokens if word not in stop_words_es]
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)



Probamos la funcion de preparacion:

In [5]:
random_text = raw_data['textos'].sample(1).item()
prepared_text = prepare_text(random_text)
print(f'Texto original:\n{random_text}\nTexto preparado:\n{prepared_text}')

Texto original:
Hay muchas definiciones de terrorismo y numerosos ejemplos del uso de explosivos y armas pequeñas, especialmente contra civiles y con el objetivo de infundir miedo. Aunque los terroristas han utilizado rara vez agentes químicos y biológicos, últimamente ha habido mucha preocupación por la amenaza del bioterrorismo y el papel del futuro personal de salud para contrarrestarlo. El establecimiento racional de prioridades requiere el equilibrio entre los riesgos y los beneficios en la prevención y la preparación. Los efectos adversos de la preparación incluyen advertencias inapropiadas, desvío de recursos de otras medidas de salud pública, tanto en los Estados Unidos como en el extranjero y restricciones a los derechos civiles. Se argumenta que EE.UU. debería contrarrestar la amenaza del bioterrorismo abordando sus causas profundas y fortaleciendo los derechos civiles, el control internacional de armas y el derecho internacional en lugar de una 'guerra contra el terrorismo' 

### Modelado LSA (Latent Semantic Analysis o Análisis Semántico Latente) 

In [6]:
# Vectorizamos los textos en el conjunto de datos
vectorizer_lsa = TfidfVectorizer(preprocessor=prepare_text)
matriz_tfidf = vectorizer_lsa.fit_transform(raw_data['textos'])

# Realizamos la reduccion de la dimencionalidad para cada ODS
num_temas = len(raw_data['ODS'].unique())
lsa_model = TruncatedSVD(n_components=num_temas, random_state=42)
matriz_lsa = lsa_model.fit_transform(matriz_tfidf)

palabras = vectorizer_lsa.get_feature_names_out()

print("--- PALABRAS CLAVE POR TEMA ---")
for i, tema in enumerate(lsa_model.components_):
    # Obtener los índices de las palabras con mayor peso en este tema
    indices_top_palabras = tema.argsort()[::-1][:10]
    palabras_tema = [palabras[idx] for idx in indices_top_palabras]
    print(f"Tema {i+1}: {', '.join(palabras_tema)}")

--- PALABRAS CLAVE POR TEMA ---
Tema 1: pais, polit, pued, desarroll, mujer, agu, trabaj, educ, gener, mejor
Tema 2: agu, energ, cost, energet, electr, gestion, renov, climat, eficient, inversion
Tema 3: derech, internacional, human, agu, articul, polit, tribunal, ley, penal, nacional
Tema 4: mujer, agu, gener, pobrez, hombr, derech, ingres, energ, trabaj, emple
Tema 5: agu, educ, salud, escuel, calid, atencion, hidric, niñ, mujer, residual
Tema 6: mujer, gener, hombr, iguald, agu, polit, educ, particip, energ, trabaj
Tema 7: salud, atencion, servici, mental, medic, pacient, mujer, primari, sanitari, segur
Tema 8: energ, derech, electr, educ, renov, escuel, energet, cost, estudi, human
Tema 9: trabaj, emple, laboral, empres, merc, product, sector, salari, tiemp, desemple
Tema 10: climat, pais, financi, ocde, inform, mujer, millon, adapt, cambi, años
Tema 11: educ, urban, desarroll, millon, años, crecimient, rural, ocde, sector, pais
Tema 12: pobrez, niñ, mujer, desarroll, financi, prog

### Interpretación de Tópicos LSA vs ODS

El Análisis Semántico Latente (LSA mediante `TruncatedSVD`) descompone la matriz TF-IDF sin conocer las etiquetas `ODS`. Por tanto, los tópicos se ordenan según la **varianza estadística explicada** y no según la numeración administrativa de la ONU:

 - **Tema 6:** `mujer, gener, hombr, iguald, polit, educ, particip, trabaj`
   * **ODS Asociado:** [ODS 5] Igualdad de género
   * **Justificación:** Las palabras dominantes giran en torno a la disparidad de género, la relación comparativa entre hombres y mujeres (`mujer`, `hombr`, `gener`), y la búsqueda de equidad y empoderamiento (`iguald`, `particip`). Refleja con total claridad las metas del ODS 5.

 - **Tema 7:** `salud, atencion, servici, mental, medic, pacient, sanitari, primari`
   * **ODS Asociado:** [ODS 3] Salud y bienestar
   * **Justificación:** El núcleo semántico reúne términos clínicos y de salud pública (`medic`, `pacient`, `atencion`, `sanitari`) con especial énfasis en servicios esenciales y salud mental (`mental`, `primari`), alineándose directamente con las coberturas universales del ODS 3.

 - **Tema 9:** `trabaj, emple, laboral, empres, merc, product, salari, desemple`
   * **ODS Asociado:** [ODS 8] Trabajo decente y crecimiento económico
   * **Justificación:** Agrupa el léxico fundamental del mercado de trabajo (`emple`, `laboral`, `desemple`), las condiciones de remuneración (`salari`) y la actividad económica empresarial y productiva (`empres`, `product`), características esenciales del ODS 8.

 - **Tema 3:** `derech, internacional, human, articul, tribunal, ley, penal, nacional`
   * **ODS Asociado:** [ODS 16] Paz, justicia e instituciones sólidas
   * **Justificación:** Concentra vocabulario jurídico e institucional internacional (`derech`, `human`, `tribunal`, `ley`, `penal`). Aborda el estado de derecho, el acceso a la justicia y los marcos normativos internacionales que promueve el ODS 16.

 - **Tema 13:** `aliment, produccion, product, pesc, alimentari, agricol, niñ`
   * **ODS Asociado:** [ODS 2] Hambre cero
   * **Justificación:** Reúne los pilares de la seguridad alimentaria mundial: producción y suministro de alimentos (`aliment`, `produccion`, `alimentari`), actividad agropecuaria y pesquera (`agricol`, `pesc`) y población vulnerable (`niñ`).


### Separacion del conjunto de datos

In [7]:
X_train, X_test, y_train, y_test = train_test_split(raw_data['textos'], raw_data['ODS'], test_size=0.2, random_state=55)

### Busqueda del mejor mocelo con cross validation

Vamos a evaluar los siguientes modelos de clasificacion para identificar cual de ellos tiene un mejor comportamineto.

 - Regresion Logistiva
 - Maquina de Soporte Vectorial
 - Gaussian NB

In [8]:
def find_best_model(X_train, y_train):
    vectorizer = TfidfVectorizer(preprocessor=prepare_text)
    tsvd = TruncatedSVD(n_components=150, random_state=32)
    # Definimos la estrategia de validación cruzada manteniendo proporción de clases
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=32)
    # 2. Definimos los modelos e hiper parametros a probar
    experimentos = {
        "Linear SVC": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", LinearSVC(max_iter=2000, random_state=32))
            ]),
            "params": {
                "model__C": [0.1, 0.5],
                "dimred__n_components": [100, 150]
            }
        },
        "Logistic Regression": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", LogisticRegression(max_iter=500, random_state=32))
            ]),
            "params": {
                "model__C": [0.5, 1.0],
                "dimred__n_components": [100]
            }
        },
        "Gaussian NB": {
            "pipeline": Pipeline([
                ("vectorizer", vectorizer),
                ("dimred", tsvd),
                ("model", GaussianNB())
            ]),
            "params": {
                "model__var_smoothing": [1e-9]
            }
        }
    }
    # Ejecutamos GridSearch para cada pipeline
    resultados = []
    mejores_modelos = {}
    for nombre, config in experimentos.items():
        print(f"Optimizando {nombre}...")
        
        # Se usa f1_weighted dado el desbalance entre los diferentes ODS
        grid = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["params"],
            cv=cv,
            scoring="f1_weighted",
            n_jobs=1,
            verbose=1
        )
        
        grid.fit(X_train, y_train)
        mejores_modelos[nombre] = grid.best_estimator_
        
        # Evaluar en el conjunto de prueba no visto
        test_score = grid.score(X_test, y_test)
        
        resultados.append({
            "Modelo": nombre,
            "Mejor F1 (CV Train)": round(grid.best_score_, 4),
            "F1 Test (X_test)": round(test_score, 4),
            "Mejores Hiperparámetros": grid.best_params_
        })
    # 4. Tabla comparativa final
    df_resultados = pd.DataFrame(resultados).sort_values(by="F1 Test (X_test)", ascending=False)
    return df_resultados

find_best_model(X_train, y_train)

Optimizando Linear SVC...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Optimizando Logistic Regression...
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Optimizando Gaussian NB...
Fitting 5 folds for each of 1 candidates, totalling 5 fits


,Modelo,Mejor F1 (CV Train),F1 Test (X_test),Mejores Hiperparámetros
0,Linear SVC,0.8631,0.8605,"{'dimred__n_components': 150, 'model__C': 0.5}"
1,Logistic Regression,0.8415,0.8340,"{'dimred__n_components': 100, 'model__C': 1.0}"
2,Gaussian NB,0.7796,0.7734,{'model__var_smoothing': 1e-09}


### Construccion del pipeline con el mejor modelo encontrado

In [9]:
steps = [
    ("vectorizer", TfidfVectorizer(preprocessor=prepare_text)),
    ("dimred", TruncatedSVD(n_components=150, random_state=32)),
    ("model", LinearSVC(C=0.5, random_state=32, max_iter=2000)),
]
pipeline = Pipeline(steps)

### Validacion de rendimiento del mejor modelo

In [10]:
# Ajuste del modelo a los datos
pipeline.fit(X_train, y_train)

#Prediccion sobre el conjunto de pruebas 
y_pred = pipeline.predict(X_test)

In [11]:
# Calculo de la matriz de confusion
ods_unicos = sorted(raw_data['ODS'].unique())
cm = confusion_matrix(y_test, y_pred, labels=ods_unicos)
pd.DataFrame(cm, index=[f"Real {i}" for i in ods_unicos], columns=[f"Pred {i}" for i in ods_unicos])

,Pred 1,Pred 2,Pred 3,Pred 4,Pred 5,Pred 6,Pred 7,Pred 8,Pred 9,Pred 10,Pred 11,Pred 12,Pred 13,Pred 14,Pred 15,Pred 16
Real 1,82,2,0,0,1,1,0,3,0,3,1,0,1,1,0,2
Real 2,2,62,4,0,0,0,1,0,4,0,2,1,1,0,1,1
Real 3,1,0,167,2,3,1,0,2,2,0,1,0,0,0,0,1
Real 4,2,0,0,189,0,0,0,1,1,2,0,0,0,0,0,1
Real 5,2,0,1,4,200,0,0,4,1,1,0,0,0,0,0,6
Real 6,0,2,1,0,0,130,1,0,0,0,0,1,5,2,1,1
Real 7,1,0,1,0,0,0,136,0,2,0,4,1,3,0,1,1
Real 8,5,0,4,5,5,0,1,49,11,1,6,0,0,0,0,1
Real 9,0,1,3,2,0,1,4,3,50,0,4,1,0,0,0,2
Real 10,9,0,1,1,1,0,0,7,0,47,2,0,1,0,0,1


In [12]:
# Reporte de clasificacion
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).transpose()

,precision,recall,f1-score,support
1,0.766355,0.845361,0.803922,97.000000
2,0.873239,0.784810,0.826667,79.000000
3,0.897849,0.927778,0.912568,180.000000
4,0.908654,0.964286,0.935644,196.000000
5,0.930233,0.913242,0.921659,219.000000
6,0.878378,0.902778,0.890411,144.000000
7,0.866242,0.906667,0.885993,150.000000
8,0.662162,0.556818,0.604938,88.000000
9,0.684932,0.704225,0.694444,71.000000
10,0.839286,0.671429,0.746032,70.000000


In [13]:
nombres_ods = {
    1: "Fin de la pobreza",
    2: "Hambre cero",
    3: "Salud y bienestar",
    4: "Educación de calidad",
    5: "Igualdad de género",
    6: "Agua limpia y saneamiento",
    7: "Energía asequible y no contaminante",
    8: "Trabajo decente y crecimiento económico",
    9: "Industria, innovación e infraestructura",
    10: "Reducción de las desigualdades",
    11: "Ciudades y comunidades sostenibles",
    12: "Producción y consumo responsables",
    13: "Acción por el clima",
    14: "Vida submarina",
    15: "Vida de ecosistemas terrestres",
    16: "Paz, justicia e instituciones sólidas"
}

# Muestra práctica de 5 textos del conjunto de prueba
muestras_indices = X_test.sample(5, random_state=39).index
preds = pipeline.predict(X_test.loc[muestras_indices])

for i, (idx, pred) in enumerate(zip(muestras_indices, preds)):
    real = y_test.loc[idx]
    print(f"--- Ejemplo {i+1} ---")
    print(f"Real:     ODS {real} - {nombres_ods.get(real, '')}")
    print(f"Predicho: ODS {pred} - {nombres_ods.get(pred, '')}")
    print(f"Texto:\n\"{X_test.loc[idx]}\"\n")


--- Ejemplo 1 ---
Real:     ODS 4 - Educación de calidad
Predicho: ODS 4 - Educación de calidad
Texto:
"Esto, a su vez, ha dado forma a la política, y la provincia se está asociando con organizaciones y proporcionando recursos para intervenir con estas tendencias. En Alberta, las encuestas de bienestar se realizan a nivel escolar y muchas escuelas las utilizan para informar sus enfoques sobre los problemas indígenas y los estudiantes. Cada año, Alberta Education produce boletas de calificaciones con 16 indicadores seleccionados para que las escuelas los usen como base para una discusión evaluativa. Co-construye el proceso para comprender la práctica en el aula en toda la escuela."

--- Ejemplo 2 ---
Real:     ODS 4 - Educación de calidad
Predicho: ODS 4 - Educación de calidad
Texto:
"Además, el seguimiento y planificación de la red escolar es limitado. Hay bastantes escuelas muy pequeñas con clases pequeñas que no ofrecen una rica experiencia de aprendizaje a los estudiantes. Esta situ

### Análisis de Resultados en el Conjunto de Prueba

* **Análisis aciertos (Ejemplos 1, 2, 3 y 4):**
  * **ODS 4 (Educación de calidad) - Ejemplos 1 y 2:** El modelo identifica con solvencia raíces léxicas determinantes como `escuel`, `estudi`, `educ` y `aprendiz`.
  * **ODS 1 (Fin de la pobreza) - Ejemplo 3:** Detecta con precisión términos como `pobrez`, `ingres` y `privacion`.
  * **ODS 5 (Igualdad de género) - Ejemplo 4:** Clasifica correctamente gracias a la coexistencia de términos comparativos de género y roles de cuidado (`madr`, `padr`, `mujer`, `hombr`, `salari`).

* **Análisis error (Ejemplo 5: ODS 8 Real vs. ODS 1 Predicho):**
  * **Justificación del error:** Aunque el texto está catalogado formalmente bajo el **ODS 8 (Trabajo decente y crecimiento económico)** por referirse al "empleo estable", contiene una fuerte carga semántica orientada al **ODS 1 (Fin de la pobreza)** debido a palabras clave con alto peso TF-IDF como `gasto social`, `riesgo de pobreza` e `independencia financiera`. Este solapamiento es natural y evidencia la **interseccionalidad semántica** entre los ODS, donde la precariedad laboral juvenil (ODS 8) es el detonante directo de la pobreza estructural (ODS 1).


# Exportar modelo

In [14]:
joblib.dump(pipeline, 'modelo_ods.joblib', compress=3)

['modelo_ods.joblib']